In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict, Optional
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [11]:
import os
hf_token = os.getenv("HUGGINGFACE_API_TOKEN")

In [12]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen3-4B-Instruct-2507",
    huggingfacehub_api_token=hf_token,
    temperature = 0.7,
    max_new_tokens=216
)
model = ChatHuggingFace(llm=llm)

In [13]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [14]:
def create_outline(state: BlogState)-> BlogState:
    # fetch title
    title = state['title']
    #call llm gen outline
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content
    #update state
    state['outline'] = outline
    
    return state

In [15]:
def create_blog(state: BlogState)-> BlogState:
    
    title = state['title']
    outline = state['outline']
    
    prompt = f'Write a detailed blog on the title{title} using the following outline \n {outline}'
    
    content = model.invoke(prompt).content
    
    state['content'] = content
    
    return state

In [17]:
graph = StateGraph(BlogState)
# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog', END)

# compile
workflow = graph.compile()

In [18]:
initial_state = {'title': 'Rise of AI in Asia'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of AI in Asia', 'outline': '**Blog Title: The Rise of AI in Asia: A Transformative Force Shaping the Future**\n\n---\n\n**I. Introduction**  \n- Brief overview of artificial intelligence (AI) as a global phenomenon  \n- Highlight Asia’s emergence as a dynamic and rapidly advancing region in AI development  \n- Purpose of the blog: To explore the key drivers, regional dynamics, technological advancements, challenges, and future implications of AI in Asia  \n- Key themes to be covered: Investment, policy, innovation, workforce impact, and global competitiveness  \n\n---\n\n**II. Why Asia is at the Forefront of AI Development**  \n- **Demographic and Economic Foundations**  \n  - Large, young, and digitally engaged populations (e.g., China, India, Southeast Asia)  \n  - Rising middle class driving demand for AI-driven services  \n- **Strong Digital Infrastructure**  \n  - Expanding 5G networks, cloud computing, and mobile penetration  \n  - Government-backed digital transf

In [19]:
print(final_state['content'])

**The Rise of AI in Asia: A Transformative Force Shaping the Future**

---

### I. Introduction

Artificial Intelligence (AI) is no longer a futuristic concept — it is a powerful, living force reshaping industries, economies, and daily life across the globe. From autonomous vehicles to predictive healthcare, AI is driving innovation at an unprecedented pace. While North America and Europe have long been seen as leaders in AI research and development, a remarkable shift is underway: **Asia is emerging as a dynamic, fast-evolving hub of AI innovation and deployment**.

Countries across the region — from China and India to South Korea, Japan, and Southeast Asian nations — are investing heavily in AI technologies, tailoring strategies to local needs, and rapidly integrating AI into governance, manufacturing, agriculture, finance, and education. This transformation is not just about adopting existing technologies — it's about building AI ecosystems from the ground up, driven by massive popu